# Transcriptome paper figures

In [7]:
from pathlib import Path
import os
import io
import re
from adjustText import adjust_text

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import h5py

from scipy.stats import pearsonr, bootstrap

from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

from IPython.display import Image, display as ipy_display

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.transforms import blended_transform_factory
from matplotlib.patches import FancyBboxPatch

matplotlib.rcParams["svg.fonttype"] = "none"
matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]

## Configuration

In [8]:
PLOT_DIR            = Path("/home/mila/l/lola.lebreton/CAP/notebooks/final_figures")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS    = ("tahoe", "lincs")

PROFILE_TYPES = ["untrt", "trt", "delta"]

# ── Tahoe metadata ─────────────────────────────────────────────────────────────────────────────────────
SAMPLE_META         = Path("/network/scratch/l/lola.lebreton/transcriptomes/final_data") / "sample_metadata.parquet"
CL_META             = Path("/network/scratch/l/lola.lebreton/transcriptomes/final_data") / "cell_line_metadata.parquet"

# ── Tahoe preprocessed data ────────────────────────────────────────────────────────────────────────────
TAHOE_PROCESSED     = Path("/network/scratch/l/lola.lebreton/transcriptomes/final_data/filtered_pseudobulks_alpha_10000.jld2")
TAHOE_DELTA         = Path("/network/scratch/l/lola.lebreton/transcriptomes/final_data/filtered_delta_alpha_10000.jld2")

# ── LINCS preprocessed data ────────────────────────────────────────────────────────────────────────────
LINCS_PROCESSED     = Path("/network/scratch/l/lola.lebreton/transcriptomes/final_data/filtered_lincs.jld2")
LINCS_DELTA         = Path("/network/scratch/l/lola.lebreton/transcriptomes/final_data/delta.jld2")

# ── PCA ────────────────────────────────────────────────────────────────────────────────────────────────
PCA_DIRS            = {ds: Path("/network/scratch/l/lola.lebreton/transcriptomes/pca") / ds for ds in DATASETS}

# ── Repro ──────────────────────────────────────────────────────────────────────────────────────────────
LINCS_REPRO_MAX_PAIRS = 200
LINCS_REPRO         = Path( "/network/scratch/l/lola.lebreton/transcriptomes/repro/lincs")
TAHOE_REPRO         = Path("/network/scratch/l/lola.lebreton/transcriptomes/repro/tahoe")

# ── Metrics ────────────────────────────────────────────────────────────────────────────────────────────
METRICS = ["pearson", "spearman", "cosine", "l2"]
METRIC_LABELS = {
    "pearson":  "Pearson r",
    "spearman": "Spearman ρ",
    "cosine":   "Cosine similarity",
    "l2":       "L2 distance",
}

SAR_PATH = {ds: Path("/network/scratch/l/lola.lebreton/transcriptomes/sar") / ds / "sar.csv" for ds in DATASETS}

## Style

In [9]:
# ── Common style ──────────────────────────────────────────────────────────
BLUE   = "#0085ff"
LBLUE  = "#67b2f8"
DBLUE  = "#003f7f"
RED    = "#d62728"
ORANGE = "#e07b00"
GREY   = "#888888"

PALETTE = {
    "tahoe": LBLUE,
    "lincs": ORANGE,
    
    "untrt": GREY,
    "trt":   RED,
    "delta": DBLUE,
    
    "train": RED,
    "val":   ORANGE,
    "test":  BLUE    
}

DS_LABELS  = {"tahoe": "Tahoe",   "lincs": "LINCS"}
SPLIT_LABELS = {"train": "Train", "val": "Validation", "test": "Test"}

# Font-size ladder — used wherever an explicit override is still needed
FS_TITLE  = 12    # panel / figure title
FS_LABEL  = 11    # axis labels
FS_TICK   = 10    # tick labels
FS_LEGEND = 11    # legend text
FS_ANNOT  = 10    # in-plot annotations
FS_SMALL  = 9    # secondary / dense annotations
FS_TINY   = 8    # very dense tick labels

mpl.rcParams.update({
    # Font
    "font.family":         "sans-serif",
    "font.sans-serif":     ["DejaVu Sans"],
    "font.size":           FS_LABEL,
    "axes.titlesize":      FS_TITLE,
    "axes.titleweight":    "bold",
    "axes.labelsize":      FS_LABEL,
    "xtick.labelsize":     FS_TICK,
    "ytick.labelsize":     FS_TICK,
    "legend.fontsize":     FS_LEGEND,
    "figure.titlesize":    FS_TITLE,
    "figure.titleweight":  "bold",
    # Axes
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.grid":           False,
    "grid.color":          "#e5e5e5",
    "grid.linewidth":      0.6,
    # Figure
    "figure.dpi":          120,
    "figure.facecolor":    "white",
    "savefig.dpi":         200,
    "savefig.bbox":        "tight",
})

HEXBIN_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "Blues_dark", plt.cm.Blues(np.linspace(0.3, 1.0, 256))
)